# Business Entity Resolution — run 02 (all three layers fixed, full training data)

Builds on the cleaned cache from run 01 (`cache/*.parquet`). Changes vs run 01 (see `runs/ITERATIONS.md`):
**Layer 3:** trains on all 2.2M train S1, so every candidate's real owner is present, as in test. **Layer 2:** transliteration table
learned from training pairs; generated-name features; website/no-space containment; name-uniqueness counts. **Layer 1:** decision
re-tuned on the new out-of-fold predictions. **Mechanics:** every stage saved per chunk (resumable); vectorised features;
parallel loops; France partitioned by region.

In [1]:
# ============================================================ 0. CONFIG
import os, re, csv, gc, sys, time, math, json, glob, warnings, subprocess
from collections import defaultdict, Counter
import numpy as np, pandas as pd, scipy.sparse as sp
from unidecode import unidecode
from rapidfuzz import fuzz
from rapidfuzz.process import cpdist
from rapidfuzz.distance import JaroWinkler, Levenshtein
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from joblib import Parallel, delayed
from sparse_dot_topn import sp_matmul_topn
import xgboost as xgb
warnings.filterwarnings("ignore")

ROOT = os.environ.get("ER_ROOT", "C:/Users/malla/Downloads/6ab10eb3b23ba_student_resource/student_resource")
DATA_DIR = f"{ROOT}/dataset"
PROJ = os.environ.get("ER_WORK", "C:/Users/malla/ml_challenghe")
CACHE_DIR = f"{PROJ}/cache"

DEV = False    # True: a few small COMPLETE states only (code check, ~15 min) | False: everything
DEV_PARTS = {"train": [("us", "ri"), ("us", "vt"), ("us", "de"), ("india", "ga"), ("india", "hp")],
             "test":  [("us", "ri"), ("us", "vt"), ("india", "ga"), ("france", "pdl")]}
RUN_DIR = f"{PROJ}/runs/" + ("run02_dev" if DEV else "run02_full")
OUT_DIR = f"{PROJ}/output_dev" if DEV else f"{PROJ}/output"
for d in (RUN_DIR, f"{RUN_DIR}/train", f"{RUN_DIR}/test", OUT_DIR): os.makedirs(d, exist_ok=True)

N_JOBS = max(1, (os.cpu_count() or 2) - 2)
SEED = 42
N_FOLDS = 3
K_TFIDF = 15               # neighbours per view per source
CAP = 40                   # candidates kept per S1 (RRF-ranked)
TRAIN_NEG_TOP = 15         # training rows: all positives + all negatives ranked < 15 ...
TRAIN_NEG_REST = 0.10      # ... + 10% of the other negatives, weighted x10 (unbiased)
CHUNK_S1 = 120_000         # S1 per saved chunk
FIT_SAMPLE = 2_000_000
T0 = time.time()
LOGF = f"{RUN_DIR}/run.log"
def log(*a):
    msg = f"[{time.strftime('%H:%M:%S')} +{time.time()-T0:6.0f}s] " + " ".join(str(x) for x in a)
    print(msg, flush=True)
    with open(LOGF, "a", encoding="utf-8") as f: f.write(msg + "\n")
log("run02 start | DEV =", DEV, "| run dir", RUN_DIR)

[14:17:23 +     0s] run02 start | DEV = False | run dir C:/Users/malla/ml_challenghe/runs/run02_full


In [2]:
# ============================================================ 1. LOAD (cleaned cache) + ground truth
def read_tsv(path):
    return pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False, quoting=csv.QUOTE_NONE, encoding="utf-8")

V2_DIR = f"{CACHE_DIR}/v2"; os.makedirs(V2_DIR, exist_ok=True)
KEYS6 = [(sp_, s) for sp_ in ("train", "test") for s in (1, 2, 3)]
HAVE_V2 = all(os.path.exists(f"{V2_DIR}/{k[0]}_s{k[1]}.parquet") for k in KEYS6)
src_dir = V2_DIR if HAVE_V2 else CACHE_DIR
PREP = {k: pd.read_parquet(f"{src_dir}/{k[0]}_s{k[1]}.parquet") for k in KEYS6}
gt = read_tsv(f"{DATA_DIR}/train/train_ground_truth.tsv")
GT = {r.source1_entity_id: [x for x in r.matched_entity_ids.split(",") if x] for r in gt.itertuples()}
OWNER = {x: s for s, v in GT.items() for x in v}
log("loaded", "v2 cache" if HAVE_V2 else "run-01 cache", {f"{k[0]}_s{k[1]}": len(v) for k, v in PREP.items()})

[14:17:48 +    25s] loaded v2 cache {'train_s1': 2206821, 'train_s2': 5034616, 'train_s3': 5285603, 'test_s1': 1732544, 'test_s2': 4887273, 'test_s3': 5082316}


In [3]:
# ============================================================ 2a. LAYER 2a: TRANSLITERATION TABLE (learned from training pairs)
# English business words written in Indic scripts come out as 'intrneshnl', 'tek', 'phuds' after transliteration.
# Align each such S2/S3 name with its S1 name (true pairs only) and keep frequent, consistent word mappings.
INDIC_RE = re.compile("[\u0900-\u0dff]")
LEGALISH = {"pvt", "ltd", "private", "limited", "llp", "inc", "co", "company", "corp", "corporation", "incorporated", "lp", "llc"}

def phon(s):
    s = re.sub(r"([a-z])\1+", r"\1", s)
    for a, b in (("ph","f"),("bh","b"),("kh","k"),("gh","g"),("dh","d"),("th","t"),("sh","s"),("ch","c"),
                 ("aa","a"),("ee","i"),("ii","i"),("oo","u"),("uu","u"),("ai","e"),("ae","e"),("ei","e"),
                 ("ey","e"),("ou","u"),("w","v"),("q","k"),("z","j"),("ck","k"),("c","k"),("d","t"),("y ","i "),
                 ("x","ks")):
        s = s.replace(a, b)
    s = re.sub(r"y$", "i", s)
    return re.sub(r"([a-z])\1+", r"\1", s)

def skel(t):
    t = phon(t)
    return t[:1] + re.sub(r"[aeiouy]", "", t[1:])

def shared_word_rate(pairs, s1d, rd):
    return np.mean([bool(set(s1d[a].split()) & set(rd[b].split())) for a, b in pairs]) if pairs else float("nan")

TRANSLIT_FILE = f"{V2_DIR}/translit.json"
s1d = dict(zip(PREP[("train", 1)]["entity_id"].values, PREP[("train", 1)]["n_core"].values))
R_tr = pd.concat([PREP[("train", s)][["entity_id", "business_name", "n_core"]] for s in (2, 3)], ignore_index=True)
is_ind = R_tr["business_name"].str.contains(INDIC_RE).values
ind_pairs = [(OWNER[r], r) for r in R_tr["entity_id"].values[is_ind] if r in OWNER]
rd_ind = dict(zip(R_tr["entity_id"].values[is_ind], R_tr["n_core"].values[is_ind]))
log(f"Indic-script S2/S3 names in train: {is_ind.sum():,} | true pairs among them: {len(ind_pairs):,} | "
    f"share with a common name word BEFORE: {shared_word_rate(ind_pairs[:200000], s1d, rd_ind):.3f}")

if HAVE_V2 and os.path.exists(TRANSLIT_FILE):
    TRANSLIT = json.load(open(TRANSLIT_FILE, encoding="utf-8"))
else:
    pair_cnt, tok_cnt = Counter(), Counter()
    for a, b in ind_pairs:
        at, bt = s1d[a].split(), rd_ind[b].split()
        if not at or not bt: continue
        if len(at) == len(bt): al = list(zip(bt, at))
        else: al = [(t, max(at, key=lambda x: Levenshtein.normalized_similarity(skel(t), skel(x)))) for t in bt]
        for t, x in al:
            tok_cnt[t] += 1
            if t != x: pair_cnt[(t, x)] += 1
    s1_vocab = Counter(t for c in s1d.values() for t in c.split())
    best = {}
    for (t, x), c in pair_cnt.items():
        if c > best.get(t, (0, None))[0]: best[t] = (c, x)
    TRANSLIT = {t: x for t, (c, x) in best.items()
                if c >= 5 and c / tok_cnt[t] >= 0.6 and x not in LEGALISH
                and Levenshtein.normalized_similarity(skel(t), skel(x)) >= 0.4 and s1_vocab[t] < 0.2 * s1_vocab[x]}
    json.dump(TRANSLIT, open(TRANSLIT_FILE, "w", encoding="utf-8"), indent=0)
log(f"transliteration table: {len(TRANSLIT):,} word mappings, e.g.",
    {k: TRANSLIT[k] for k in ["intrneshnl", "tek", "phuds", "enrji", "sistms", "knsltensi", "globl", "treding"] if k in TRANSLIT})

def apply_translit(core):
    toks = [TRANSLIT.get(t, t) for t in core.split()]
    return " ".join([t for t in toks if t not in LEGALISH] or toks)

if not HAVE_V2:
    for k, df in PREP.items():
        m = df["business_name"].str.contains(INDIC_RE).values
        if not m.any(): continue
        core = [apply_translit(c) for c in df["n_core"].values[m]]
        full = [" ".join(TRANSLIT.get(t, t) for t in f.split()) for f in df["n_full"].values[m]]
        df.loc[m, "n_core"] = core
        df.loc[m, "n_full"] = full
        df.loc[m, "n_phon"] = [phon(c) for c in core]
        df.loc[m, "n_nospace"] = [c.replace(" ", "") for c in core]
        df.loc[m, "all_txt"] = [(x + " " + y).strip() for x, y in zip(full, df["a_full"].values[m])]
        df.loc[m, "all_phon"] = [(phon(x) + " " + y).strip() for x, y in zip(core, df["a_phon"].values[m])]
        df.loc[m, "n_empty"] = np.array([len(x) == 0 for x in core], np.int8)
        log(f"   {k}: transliteration applied to {m.sum():,} Indic-script names")
rd_after = dict(zip(pd.concat([PREP[("train", s)] for s in (2, 3)])["entity_id"].values,
                    pd.concat([PREP[("train", s)] for s in (2, 3)])["n_core"].values))
s1d = dict(zip(PREP[("train", 1)]["entity_id"].values, PREP[("train", 1)]["n_core"].values))
log(f"share of Indic true pairs with a common name word AFTER: {shared_word_rate(ind_pairs[:200000], s1d, rd_after):.3f}")
del R_tr, rd_ind, rd_after; gc.collect()

[14:17:54 +    31s] Indic-script S2/S3 names in train: 752,869 | true pairs among them: 551,240 | share with a common name word BEFORE: 0.994
[14:17:54 +    31s] transliteration table: 503 word mappings, e.g. {'intrneshnl': 'intl', 'tek': 'tech', 'phuds': 'fods', 'enrji': 'energy', 'sistms': 'systems', 'knsltensi': 'consultancy', 'globl': 'global', 'treding': 'trading'}
[14:18:18 +    56s] share of Indic true pairs with a common name word AFTER: 0.994


30

In [4]:
# ============================================================ 2b. PARTITION KEY (state group; France: region)
STATE_GROUP = {"tg": "tg_ap", "ap": "tg_ap"}
FR_REG = {"hauts de france": "hdf", "nord": "hdf", "pas de calais": "hdf", "nouvelle aquitaine": "naq", "gironde": "naq",
          "pays de la loire": "pdl", "loire atlantique": "pdl", "ile de france": "idf", "occitanie": "occ",
          "bretagne": "bre", "normandie": "nor", "grand est": "ges", "auvergne rhone alpes": "ara",
          "provence alpes cote d azur": "pac", "centre val de loire": "cvl", "bourgogne franche comte": "bfc", "corse": "cor"}
def _fr_comps(addr): return [" ".join(re.sub(r"[^a-z ]", " ", unidecode(c).lower()).split()) for c in addr.split(",")]
def fr_region(comps):
    for c in reversed(comps):
        if c in FR_REG: return FR_REG[c]
    return ""

if not HAVE_V2:
    # learn city -> region from French S1 records (they always carry the region)
    s1f = PREP[("test", 1)].query("country_n == 'france'")["business_address"].values
    cc = defaultdict(Counter)
    for a in s1f:
        comps = _fr_comps(a); reg = fr_region(comps)
        if reg:
            for c in comps:
                if c and c not in FR_REG and not re.search(r"\d", c): cc[c][reg] += 1
    CITY_REG = {c: r.most_common(1)[0][0] for c, r in cc.items()
                if sum(r.values()) >= 3 and r.most_common(1)[0][1] / sum(r.values()) >= 0.95}
    json.dump(CITY_REG, open(f"{V2_DIR}/fr_city_region.json", "w"), indent=0)
    log(f"France: {len(CITY_REG):,} cities mapped to a region (learned from S1)")
    for k, df in PREP.items():
        part = np.array([STATE_GROUP.get(s, s) for s in df["a_state"].values], dtype=object)
        fr = df["country_n"].values == "france"
        if fr.any():
            vals = []
            for a in df["business_address"].values[fr]:
                comps = _fr_comps(a); reg = fr_region(comps)
                if not reg:
                    reg = next((CITY_REG[c] for c in comps if c in CITY_REG), "")
                vals.append(reg)
            part[fr] = vals
            log(f"   {k}: France records with a region: {np.mean([v != '' for v in vals]):.3f}")
        df["a_part"] = part
    for k, df in PREP.items():
        df.to_parquet(f"{V2_DIR}/{k[0]}_s{k[1]}.parquet", index=False)
    log("v2 cache written")

In [5]:
# ============================================================ 3. L / R PER SPLIT + WHOLE-DATA STATISTICS (layer 2b/2d)
LR, LALL = {}, {}
for sp_ in ("train", "test"):
    L_all = PREP.pop((sp_, 1)).reset_index(drop=True)
    R = pd.concat([PREP.pop((sp_, 2)), PREP.pop((sp_, 3))], ignore_index=True)
    R["src"] = np.where(R["entity_id"].str.startswith("S2").values, 2, 3).astype(np.int8)
    LALL[sp_] = L_all
    LR[sp_] = R
gc.collect()

def _grams(t):
    t = f"^{t}$"; return [t[i:i+3] for i in range(len(t) - 2)]

# generated-name model: words of S2/S3 names that share nothing with their true S1 name and never occur in real S1 names
s1_vocab = {sp_: Counter(t for c in LALL[sp_]["n_core"].values for t in c.split()) for sp_ in LALL}
real_vocab = set(s1_vocab["train"]) | set(s1_vocab["test"])
Rt = LR["train"]; s1core = dict(zip(LALL["train"]["entity_id"].values, LALL["train"]["n_core"].values))
WEB_NAME_RE = re.compile(r"\.(?:com|net|org|in|fr|co|biz)|www\.", re.I)
gen_toks = Counter()
for rid, core, nm in zip(Rt["entity_id"].values, Rt["n_core"].values, Rt["business_name"].values):
    s = OWNER.get(rid)
    if not s or INDIC_RE.search(nm) or WEB_NAME_RE.search(nm): continue   # websites glue words -> not "generated"
    toks = core.split()
    if set(toks) & set(s1core[s].split()): continue
    for t in toks:
        if 4 <= len(t) <= 14 and t.isalpha() and t not in real_vocab: gen_toks[t] += 1
gG = Counter(g for t in gen_toks for g in _grams(t))
gS = Counter(g for t in real_vocab if len(t) >= 4 and t.isalpha() for g in _grams(t))
nG, nS, V = sum(gG.values()), sum(gS.values()), len(set(gG) | set(gS))
def gen_score(t):
    if len(t) < 4 or not t.isalpha(): return -5.0
    gs = _grams(t)
    return float(np.mean([math.log((gG[g] + 1) / (nG + V)) - math.log((gS[g] + 1) / (nS + V)) for g in gs]))
log(f"generated-name model: {len(gen_toks):,} generated words, e.g. {[t for t, _ in gen_toks.most_common(8)]}")
log("   scores:", {w: round(gen_score(w), 2) for w in ["zephzeph", "brixumbra", "miralyrafaye", "barbershop", "finance", "logistics", "ecole"]})

_gcache = {}
def _gmax(core):
    best = -5.0
    for t in core.split():
        v = _gcache.get(t)
        if v is None: v = _gcache[t] = gen_score(t)
        if v > best: best = v
    return best

def add_row_stats(sp_):
    L, R = LALL[sp_], LR[sp_]
    s1name = (L["country_n"] + "|" + L["n_core"]).value_counts()
    s1first = (L["country_n"] + "|" + L["n_core"].str.split(n=1).str[0].fillna("")).value_counts()
    rname = (R["country_n"] + "|" + R["n_core"]).value_counts()
    L["l_name_n"] = (L["country_n"] + "|" + L["n_core"]).map(s1name).fillna(0).astype(np.float32).values
    L["l_first_n"] = (L["country_n"] + "|" + L["n_core"].str.split(n=1).str[0].fillna("")).map(s1first).fillna(0).astype(np.float32).values
    R["r_name_n_s1"] = (R["country_n"] + "|" + R["n_core"]).map(s1name).fillna(0).astype(np.float32).values
    R["r_name_n_r"] = (R["country_n"] + "|" + R["n_core"]).map(rname).fillna(0).astype(np.float32).values
    voc = s1_vocab[sp_]
    oov = [sum(1 for t in c.split() if voc[t] == 0) for c in R["n_core"].values]
    ntok = R["n_core"].str.count(" ").values + (R["n_core"].str.len().values > 0)
    R["r_oov_n"] = np.array(oov, np.float32)
    R["r_oov_frac"] = np.where(ntok > 0, np.array(oov) / np.maximum(ntok, 1), 0).astype(np.float32)
    R["r_gen_max"] = np.array([_gmax(c) for c in R["n_core"].values], np.float32)
    L["l_gen_max"] = np.array([_gmax(c) for c in L["n_core"].values], np.float32)
    R["r_is_web"] = R["business_name"].str.lower().str.contains(r"\.(?:com|net|org|in|fr|co|biz)\b|www\.", regex=True).astype(np.int8).values

for sp_ in ("train", "test"):
    t = time.time(); add_row_stats(sp_); log(f"{sp_}: whole-data row statistics added ({time.time()-t:.0f}s)")

def pick_L(sp_):
    L = LALL[sp_]
    if not DEV: return L
    keep = np.zeros(len(L), bool)
    for c, p in DEV_PARTS[sp_]:
        keep |= (L["country_n"].values == c) & (L["a_part"].values == p)
    return L[keep].reset_index(drop=True)
LSUB = {sp_: pick_L(sp_) for sp_ in ("train", "test")}
log({sp_: f"L={len(LSUB[sp_]):,} R={len(LR[sp_]):,}" for sp_ in LSUB})

[14:19:20 +   117s] generated-name model: 209,315 generated words, e.g. ['earnosethroat', 'earnose', 'revitup', 'fotankle', 'womenshealth', 'primarycare', 'urgentcare', 'mainstret']
[14:19:20 +   117s]    scores: {'zephzeph': 2.38, 'brixumbra': 1.37, 'miralyrafaye': 1.5, 'barbershop': -0.1, 'finance': 0.01, 'logistics': 0.22, 'ecole': 0.05}
[14:22:18 +   295s] train: whole-data row statistics added (178s)
[14:23:32 +   369s] test: whole-data row statistics added (74s)
[14:23:32 +   369s] {'train': 'L=2,206,821 R=10,320,219', 'test': 'L=1,732,544 R=9,969,589'}


In [6]:
# ============================================================ 4. BLOCKING (views, partitions, exact keys, RRF)
VIEWS = {  # name: (column, analyzer, ngram range, max_df)
    "v_name_c": ("n_core",   "char_wb", (3, 4), 0.02),
    "v_addr_c": ("a_street", "char_wb", (3, 4), 0.02),
    "v_all_w":  ("all_txt",  "word",    (1, 2), 0.01),
    "v_phon_c": ("all_phon", "char_wb", (3, 4), 0.02),
}
KEY_VIEWS = ["v_key_addr", "v_key_name"]
ALL_V = list(VIEWS) + KEY_VIEWS
KEY_MAX_BUCKET = 50

def _to_np(s): return s.to_numpy(dtype=object)

def _transform(vec, texts, step=400_000):
    parts = Parallel(n_jobs=min(N_JOBS, 8))(delayed(vec.transform)(texts[i:i+step]) for i in range(0, len(texts), step))
    return sp.vstack(parts).tocsr()

def fit_views(L_all, L, R):
    mats = {}
    rs = R.sample(min(len(R), FIT_SAMPLE), random_state=SEED)
    for v, (col, an, ng, mx) in VIEWS.items():
        t = time.time()
        vec = TfidfVectorizer(analyzer=an, ngram_range=ng, min_df=2, max_df=mx, sublinear_tf=True, dtype=np.float32)
        vec.fit(np.concatenate([_to_np(L_all[col]), _to_np(rs[col])]))
        mats[v] = (_transform(vec, _to_np(L[col])), _transform(vec, _to_np(R[col])))
        log(f"   view {v}: vocab={len(vec.vocabulary_):,} ({time.time()-t:.0f}s)")
    return mats

def make_chunks(L, R):
    """list of (country, [(part, lidx, ridx), ...]) with <= CHUNK_S1 S1 per chunk, never mixing countries"""
    Lc, Lp = _to_np(L["country_n"]), _to_np(L["a_part"]); Rc, Rp = _to_np(R["country_n"]), _to_np(R["a_part"])
    chunks = []
    for c in np.unique(Lc):
        lc, rc = Lc == c, Rc == c
        r_nopart = rc & (Rp == "")
        cur, n = [], 0
        for p in np.unique(Lp[lc]):
            lidx = np.flatnonzero(lc & (Lp == p))
            ridx = np.flatnonzero(rc if p == "" else ((rc & (Rp == p)) | r_nopart))
            for st in range(0, len(lidx), CHUNK_S1):
                sub = lidx[st:st + CHUNK_S1]
                if cur and n + len(sub) > CHUNK_S1:
                    chunks.append((c, cur)); cur, n = [], 0
                cur.append((p, sub, ridx)); n += len(sub)
        if cur: chunks.append((c, cur))
    return chunks

def _addr_key(df):
    num0 = df["a_nums"].str.split(n=1).str[0].fillna("")
    w0 = df["a_words"].str.split(n=1).str[0].fillna("")
    k = (df["country_n"] + "|" + num0 + "|" + w0).to_numpy(dtype=object)
    k[(num0 == "").values | (w0 == "").values] = ""
    return k

def _name_key(df, dfreq):
    out = []
    for c, core in zip(df["country_n"].values, df["n_core"].values):
        toks = [t for t in core.split() if len(t) >= 3]
        out.append(c + "|" + min(toks, key=lambda t: dfreq.get(t, 0)) if toks else "")
    return np.array(out, dtype=object)

def build_keys(L_all, L, R):
    dfreq = pd.concat([L_all["n_core"], R["n_core"]]).str.split().explode().value_counts().to_dict()
    keys = {}
    for v, fn in (("v_key_addr", _addr_key), ("v_key_name", lambda d: _name_key(d, dfreq))):
        rk = fn(R); rdf = pd.DataFrame({"k": rk, "ri": np.arange(len(rk), dtype=np.int32)}); rdf = rdf[rdf.k != ""]
        rdf = rdf[rdf["k"].map(rdf["k"].value_counts()).values <= KEY_MAX_BUCKET]
        keys[v] = (fn(L), rdf)
    return keys

def block_chunk(parts, L, R, mats, keys):
    nR, src_r = len(R), R["src"].values
    vid = {v: i for i, v in enumerate(ALL_V)}
    LI, RI, SC, VW = [], [], [], []
    for p, lidx, ridx in parts:
        for src in (2, 3):
            rs = ridx[src_r[ridx] == src]
            if len(rs) == 0: continue
            for v in VIEWS:
                A_, B_ = mats[v]
                C = sp_matmul_topn(A_[lidx], B_[rs].T.tocsr(), top_n=K_TFIDF, threshold=1e-6, n_threads=N_JOBS).tocoo()
                LI.append(lidx[C.row]); RI.append(rs[C.col]); SC.append(C.data.astype(np.float32))
                VW.append(np.full(len(C.data), vid[v], np.int8))
    lsel = np.concatenate([x[1] for x in parts])
    for v, (lk, rdf) in keys.items():
        ldf = pd.DataFrame({"k": lk[lsel], "li": lsel}); ldf = ldf[ldf.k != ""]
        m = ldf.merge(rdf, on="k")
        LI.append(m["li"].values); RI.append(m["ri"].values); SC.append(np.ones(len(m), np.float32))
        VW.append(np.full(len(m), vid[v], np.int8))
    li, ri, sc, vw = map(np.concatenate, (LI, RI, SC, VW))
    key = li.astype(np.int64) * nR + ri
    uk, inv = np.unique(key, return_inverse=True)
    W = np.full((len(uk), len(ALL_V)), np.nan, np.float32); W[inv, vw] = sc
    P = pd.DataFrame(W, columns=ALL_V)
    P.insert(0, "li", (uk // nR).astype(np.int32)); P.insert(1, "ri", (uk % nR).astype(np.int32))
    P["n_views"] = np.isfinite(W).sum(1).astype(np.int8)
    P["bsum"] = np.nan_to_num(W).sum(1)
    src = src_r[P["ri"].values]
    rrf = np.zeros(len(P))
    for v in VIEWS:
        s = P[v].fillna(-1).values
        r = pd.Series(s).groupby([P["li"].values, src]).rank(ascending=False, method="first").values
        rrf += np.where(s > 0, 1.0 / (5 + r), 0)
    rrf += 0.2 * P["v_key_addr"].notna().values + 0.1 * P["v_key_name"].notna().values
    P["brrf"] = rrf.astype(np.float32)
    P = P.sort_values(["li", "brrf"], ascending=[True, False]).reset_index(drop=True)
    P["brank"] = P.groupby("li").cumcount().values.astype(np.int16)
    return P[P["brank"].values < CAP].reset_index(drop=True)
log("blocking functions defined")

[14:23:32 +   369s] blocking functions defined


In [7]:
# ============================================================ 5. FEATURES (vectorised)
def rowdot(A, B, li, ri, bs=1_000_000):
    out = np.zeros(len(li), np.float32)
    for st in range(0, len(li), bs):
        out[st:st+bs] = np.asarray(A[li[st:st+bs]].multiply(B[ri[st:st+bs]]).sum(1)).ravel()
    return out

def rowmax(M):
    return np.asarray(M.max(axis=1).todense()).ravel() if M.shape[0] else np.zeros(0)

def rowmax_prod(A, B, li, ri, bs=1_000_000):
    out = np.zeros(len(li), np.float32)
    for st in range(0, len(li), bs):
        out[st:st+bs] = rowmax(A[li[st:st+bs]].multiply(B[ri[st:st+bs]]).tocsr())
    return out

def rowmax_diff(A, B, li, ri, bs=1_000_000):
    out = np.zeros(len(li), np.float32)
    for st in range(0, len(li), bs):
        D = (A[li[st:st+bs]] - B[ri[st:st+bs]]).tocsr(); D.data = np.abs(D.data)
        out[st:st+bs] = rowmax(D)
    return out

def build_tok(L_all, L, R):
    """sparse token matrices: sqrt(idf)-weighted names / address words, binary house numbers, first number"""
    T = {}
    for nm, col in (("n", "n_core"), ("a", "a_words")):
        vec = TfidfVectorizer(token_pattern=r"\S+", lowercase=False, binary=True, norm=None, dtype=np.float32)
        vec.fit(np.concatenate([_to_np(L_all[col]), _to_np(R[col])]))
        Lm, Rm = _transform(vec, _to_np(L[col])), _transform(vec, _to_np(R[col]))
        Lm.data = np.sqrt(Lm.data); Rm.data = np.sqrt(Rm.data)
        T[nm] = (Lm, Rm, np.asarray(Lm.multiply(Lm).sum(1)).ravel(), np.asarray(Rm.multiply(Rm).sum(1)).ravel())
    vec = CountVectorizer(token_pattern=r"\S+", lowercase=False, binary=True, dtype=np.float32)
    vec.fit(np.concatenate([_to_np(L_all["a_nums"]), _to_np(R["a_nums"])]))
    first = lambda d: _to_np(d["a_nums"].str.split(n=1).str[0].fillna(""))
    T["num"] = (_transform(vec, _to_np(L["a_nums"])), _transform(vec, _to_np(R["a_nums"])))
    T["first"] = (_transform(vec, first(L)), _transform(vec, first(R)), first(L), first(R))
    return T

def pair_cols(L, R, li, ri, col_l, col_r=None):
    return L[col_l].values[li].tolist(), R[col_r or col_l].values[ri].tolist()

def base_features(L, R, P, mats, T):
    li, ri = P["li"].values, P["ri"].values
    F = pd.DataFrame(index=P.index)
    for c in ALL_V + ["n_views", "bsum", "brrf", "brank"]:
        F["blk_" + c] = P[c].values
    for v in VIEWS:
        F["cos_" + v] = rowdot(mats[v][0], mats[v][1], li, ri)
    W = dict(workers=-1, dtype=np.float32)
    a, b = pair_cols(L, R, li, ri, "n_core")
    F["n_ratio"] = cpdist(a, b, scorer=fuzz.ratio, **W)
    F["n_tset"] = cpdist(a, b, scorer=fuzz.token_set_ratio, **W)
    F["n_tsort"] = cpdist(a, b, scorer=fuzz.token_sort_ratio, **W)
    F["n_partial"] = cpdist(a, b, scorer=fuzz.partial_ratio, **W)
    F["n_jw"] = cpdist(a, b, scorer=JaroWinkler.normalized_similarity, **W)
    F["n_lev"] = cpdist(a, b, scorer=Levenshtein.distance, **W)
    a2, b2 = pair_cols(L, R, li, ri, "n_full"); F["n_full_tset"] = cpdist(a2, b2, scorer=fuzz.token_set_ratio, **W)
    a2, b2 = pair_cols(L, R, li, ri, "n_phon"); F["n_phon_ratio"] = cpdist(a2, b2, scorer=fuzz.token_sort_ratio, **W)
    na, nb = pair_cols(L, R, li, ri, "n_nospace")
    F["n_nospace_ratio"] = cpdist(na, nb, scorer=fuzz.ratio, **W)
    F["n_nospace_partial"] = cpdist(na, nb, scorer=fuzz.partial_ratio, **W)
    F["n_contain"] = np.array([(len(x) >= 5 and len(y) >= 5 and (x in y or y in x)) for x, y in zip(na, nb)], np.int8)  # layer 2c
    a, b = pair_cols(L, R, li, ri, "a_street")
    F["a_ratio"] = cpdist(a, b, scorer=fuzz.ratio, **W)
    F["a_tset"] = cpdist(a, b, scorer=fuzz.token_set_ratio, **W)
    F["a_tsort"] = cpdist(a, b, scorer=fuzz.token_sort_ratio, **W)
    F["a_partial"] = cpdist(a, b, scorer=fuzz.partial_ratio, **W)
    a2, b2 = pair_cols(L, R, li, ri, "a_phon"); F["a_phon_tsort"] = cpdist(a2, b2, scorer=fuzz.token_sort_ratio, **W)
    a2, b2 = pair_cols(L, R, li, ri, "all_txt")
    F["all_tset"] = cpdist(a2, b2, scorer=fuzz.token_set_ratio, **W)
    F["all_tsort"] = cpdist(a2, b2, scorer=fuzz.token_sort_ratio, **W)
    a2, b2 = pair_cols(L, R, li, ri, "n_core", "a_street"); F["x_name_addr"] = cpdist(a2, b2, scorer=fuzz.token_set_ratio, **W)
    a2, b2 = pair_cols(L, R, li, ri, "a_street", "n_core"); F["x_addr_name"] = cpdist(a2, b2, scorer=fuzz.token_set_ratio, **W)
    # idf-weighted token overlaps (sparse, vectorised)
    for nm, pre in (("n", "n"), ("a", "a")):
        Lm, Rm, sL, sR = T[nm]
        inter = rowdot(Lm, Rm, li, ri)
        union = sL[li] + sR[ri] - inter
        F[f"{pre}_idf_jac"] = np.where(union > 0, inter / np.maximum(union, 1e-9), 0).astype(np.float32)
        F[f"{pre}_idf_maxshared"] = rowmax_prod(Lm, Rm, li, ri)
        if nm == "n":
            F["n_idf_maxunshared"] = rowmax_diff(Lm, Rm, li, ri) ** 2
            F["n_ntok_l"] = Lm.getnnz(1)[li].astype(np.float32); F["n_ntok_r"] = Rm.getnnz(1)[ri].astype(np.float32)
        else:
            F["a_nwords_diff"] = np.abs(Lm.getnnz(1)[li] - Rm.getnnz(1)[ri]).astype(np.float32)
    NL, NR = T["num"]; FL, FR, fl, fr = T["first"]
    nl, nr = NL.getnnz(1)[li], NR.getnnz(1)[ri]
    shared = rowdot(NL, NR, li, ri)
    F["num_shared"] = shared
    F["num_jac"] = np.where((nl + nr) > 0, shared / np.maximum(nl + nr - shared, 1), -1).astype(np.float32)
    F["num_symdiff"] = (nl + nr - 2 * shared).astype(np.float32)
    has = (fl[li] != "") & (fr[ri] != "")
    F["num_first_eq"] = np.where(has, (fl[li] == fr[ri]).astype(np.float32), -1)
    fin = (rowdot(FL, NR, li, ri) > 0) | (rowdot(NL, FR, li, ri) > 0)
    F["num_first_in"] = np.where(has, fin.astype(np.float32), -1)
    def eq(col):
        x, y = _to_np(L[col])[li], _to_np(R[col])[ri]
        return np.where((x != "") & (y != ""), (x == y).astype(np.float32), -1).astype(np.float32)
    F["zip_eq"] = eq("a_zip"); F["state_eq"] = eq("a_state"); F["legal_eq"] = eq("n_legal")
    f1 = lambda d: _to_np(d["n_core"].str.split(n=1).str[0].fillna(""))
    x, y = f1(L)[li], f1(R)[ri]
    F["n_first_eq"] = ((x == y) & (x != "")).astype(np.int8)
    F["src"] = R["src"].values[ri]
    F["l_a_empty"] = L["a_empty"].values[li]; F["r_a_empty"] = R["a_empty"].values[ri]
    F["r_n_empty"] = R["n_empty"].values[ri]; F["r_alias"] = R["n_alias"].values[ri]
    F["len_n_r"] = R["n_core"].str.len().values[ri]; F["len_n_l"] = L["n_core"].str.len().values[li]
    # layer 2b/2d: generated-name and uniqueness statistics (whole-data)
    for c in ("l_name_n", "l_first_n", "l_gen_max"): F[c] = L[c].values[li]
    for c in ("r_name_n_s1", "r_name_n_r", "r_oov_n", "r_oov_frac", "r_gen_max", "r_is_web"): F[c] = R[c].values[ri]
    F["combo"] = F[["cos_v_name_c", "cos_v_addr_c", "cos_v_all_w", "cos_v_phon_c"]].mean(1)
    le_n = L["n_core"].str.len().values[li] == 0; re_n = R["n_core"].str.len().values[ri] == 0
    le_a = L["a_street"].str.len().values[li] == 0; re_a = R["a_street"].str.len().values[ri] == 0
    for c in ["n_ratio","n_tset","n_tsort","n_partial","n_jw","n_lev","n_full_tset","n_phon_ratio",
              "n_nospace_ratio","n_nospace_partial","x_name_addr","n_idf_jac"]:
        F.loc[le_n | re_n, c] = np.nan
    for c in ["a_ratio","a_tset","a_tsort","a_partial","a_phon_tsort","x_addr_name","a_idf_jac","cos_v_addr_c"]:
        F.loc[le_a | re_a, c] = np.nan
    F["both_a_empty"] = (le_a & re_a).astype(np.int8); F["both_n_empty"] = (le_n & re_n).astype(np.int8)
    return F
log("feature functions defined")

[14:23:32 +   369s] feature functions defined


In [8]:
# ============================================================ 6. STAGE A: blocking + base features per chunk (saved, resumable)
def gt_keys(L, R):
    rpos = pd.Index(R["entity_id"].values)
    gl, gr = [], []
    for l, s in enumerate(L["entity_id"].values):
        for x in GT.get(s, ()): gl.append(l); gr.append(x)
    gi = rpos.get_indexer(gr); ok = gi >= 0
    return np.array(gl, np.int64)[ok] * len(R) + gi[ok].astype(np.int64), np.bincount(np.array(gl, np.int64), minlength=len(L))

def stage_A(sp_):
    L, R, L_all = LSUB[sp_], LR[sp_], LALL[sp_]
    chunks = make_chunks(L, R)
    todo = [i for i in range(len(chunks)) if not os.path.exists(f"{RUN_DIR}/{sp_}/A_{i:04d}.parquet")]
    log(f"{sp_}: {len(chunks)} chunks, {len(todo)} to do")
    json.dump([c for c, _ in chunks], open(f"{RUN_DIR}/{sp_}/chunk_country.json", "w"))
    if not todo: return
    mats = fit_views(L_all, L, R); keys = build_keys(L_all, L, R); T = build_tok(L_all, L, R)
    GK = gt_keys(L, R)[0] if sp_ == "train" else None
    for i in todo:
        t = time.time(); c, parts = chunks[i]
        P = block_chunk(parts, L, R, mats, keys)
        F = base_features(L, R, P, mats, T)
        F.insert(0, "li", P["li"].values); F.insert(1, "ri", P["ri"].values)
        if GK is not None:
            F["y"] = np.isin(P["li"].values.astype(np.int64) * len(R) + P["ri"].values, GK).astype(np.int8)
        F.to_parquet(f"{RUN_DIR}/{sp_}/A_{i:04d}.parquet", index=False)
        nS1 = sum(len(x[1]) for x in parts)
        log(f"   {sp_} A_{i:04d} [{c}] S1={nS1:,} pairs={len(F):,}" + (f" pos={F['y'].sum():,}" if GK is not None else "")
            + f" ({time.time()-t:.0f}s)")
        del P, F; gc.collect()
    del mats, keys, T; gc.collect()

for sp_ in ("train", "test"):
    stage_A(sp_)
TRUE_N = gt_keys(LSUB["train"], LR["train"])[1].astype(np.float64)
A_files = lambda sp_: sorted(glob.glob(f"{RUN_DIR}/{sp_}/A_*.parquet"))
pos = sum(pd.read_parquet(f, columns=["y"])["y"].sum() for f in A_files("train"))
log(f"train blocking recall (top {CAP}): {pos/TRUE_N.sum():.4f}  ({pos:,}/{int(TRUE_N.sum()):,})")

[14:16:22 +   269s] train: 2 chunks, 0 to do
[14:16:24 +   272s] test: 3 chunks, 0 to do
[14:16:30 +   277s] train blocking recall (top 40): 0.9907  (50,500/50,976)


In [9]:
# ============================================================ 7. STAGE B: competition features per country (saved)
COMP_COLS = ["combo", "cos_v_name_c", "cos_v_addr_c", "n_tset", "a_tset"]

def second_max(x, key):
    x = np.asarray(x, np.float64); key = np.asarray(key)
    o = np.lexsort((-x, key)); ks = key[o]
    first = np.r_[True, ks[1:] != ks[:-1]]
    pos = np.arange(len(o)) - np.maximum.accumulate(np.where(first, np.arange(len(o)), 0))
    sec_val = pd.Series(np.where(pos == 1, x[o], -np.inf)).groupby(ks).transform("max").values
    out = np.empty(len(x)); out[o] = np.where(np.isinf(sec_val), -1, sec_val)
    return out

def stage_B(sp_):
    files = A_files(sp_); cc = json.load(open(f"{RUN_DIR}/{sp_}/chunk_country.json"))
    for c in sorted(set(cc)):
        idx = [i for i, x in enumerate(cc) if x == c]
        outs = [f"{RUN_DIR}/{sp_}/B_{i:04d}.parquet" for i in idx]
        if all(os.path.exists(o) for o in outs): continue
        t = time.time()
        D = [pd.read_parquet(files[i], columns=["li", "ri", "src"] + COMP_COLS) for i in idx]
        n = [len(d) for d in D]; D = pd.concat(D, ignore_index=True)
        li, ri = D["li"], D["ri"]
        G = pd.DataFrame(index=D.index)
        for col in COMP_COLS:
            x = D[col]
            G[f"{col}_rank_l"] = x.groupby([li, D["src"]]).rank(ascending=False, method="min").astype(np.float32)
            G[f"{col}_gap_l"] = (x - x.groupby(li).transform("max")).astype(np.float32)
            mx = x.groupby(ri).transform("max"); sec = second_max(x.fillna(-1).values, ri.values)
            G[f"{col}_gap_r"] = np.where(x >= mx, x - sec, x - mx).astype(np.float32)
            G[f"{col}_rank_r"] = x.groupby(ri).rank(ascending=False, method="min").astype(np.float32)
        G["n_cand"] = li.map(li.value_counts()).astype(np.float32).values
        G["r_n_s1"] = ri.map(ri.value_counts()).astype(np.float32).values
        st = 0
        for o, k in zip(outs, n):
            G.iloc[st:st + k].reset_index(drop=True).to_parquet(o, index=False); st += k
        log(f"   {sp_} stage B [{c}]: {len(D):,} pairs ({time.time()-t:.0f}s)")
        del D, G; gc.collect()

for sp_ in ("train", "test"): stage_B(sp_)
log("stage B done")

[14:16:30 +   277s] stage B done


In [10]:
# ============================================================ 8. STAGE 1 MODEL (XGBoost on GPU, all S1, weighted negative subsampling)
GBM_PARAMS = dict(objective="binary:logistic", eval_metric="logloss", tree_method="hist", device="cuda",
                  learning_rate=0.08, grow_policy="lossguide", max_leaves=127, max_depth=0, subsample=0.8,
                  colsample_bytree=0.7, reg_lambda=2.0, min_child_weight=5, max_bin=256, seed=SEED)
PRED_CHUNK = 2_000_000
FOLD = np.random.default_rng(SEED).permutation(len(LSUB["train"])) % N_FOLDS   # fold of each train S1

def load_chunk(sp_, i, extra=()):
    parts = [pd.read_parquet(f"{RUN_DIR}/{sp_}/A_{i:04d}.parquet"), pd.read_parquet(f"{RUN_DIR}/{sp_}/B_{i:04d}.parquet")]
    for e in extra: parts.append(pd.read_parquet(f"{RUN_DIR}/{sp_}/{e}_{i:04d}.parquet"))
    return pd.concat(parts, axis=1)

class _Iter(xgb.DataIter):
    def __init__(self, parts): self.parts, self.i = parts, 0; super().__init__()
    def next(self, input_data):
        if self.i == len(self.parts): return 0
        X, y, w = self.parts[self.i]; input_data(data=X, label=y, weight=w); self.i += 1; return 1
    def reset(self): self.i = 0

def build_train_matrix(feats, extra=()):
    Xs, ys, ws, fs = [], [], [], []
    for i in range(len(A_files("train"))):
        D = load_chunk("train", i, extra)
        rng = np.random.default_rng(SEED + i)
        y_ = D["y"].values; rank = D["blk_brank"].values
        rest = (y_ == 0) & (rank >= TRAIN_NEG_TOP)
        sel = ~rest | (rng.random(len(D)) < TRAIN_NEG_REST)
        Xs.append(D.loc[sel, feats].to_numpy(np.float32)); ys.append(y_[sel])
        ws.append(np.where(rest[sel], 1 / TRAIN_NEG_REST, 1.0).astype(np.float32)); fs.append(FOLD[D["li"].values[sel]])
        del D; gc.collect()
    X, y, w, f = np.vstack(Xs), np.concatenate(ys), np.concatenate(ws), np.concatenate(fs)
    o = np.argsort(f, kind="stable")                      # contiguous fold blocks -> no copies per fold
    return X[o], y[o], w[o], f[o]

def train_folds(X, y, w, f, feats, params=GBM_PARAMS, rounds=4000):
    models = []
    bounds = [np.searchsorted(f, k) for k in range(N_FOLDS + 1)]
    for k in range(N_FOLDS):
        t = time.time()
        tr_parts = [(X[bounds[j]:bounds[j+1]], y[bounds[j]:bounds[j+1]], w[bounds[j]:bounds[j+1]]) for j in range(N_FOLDS) if j != k]
        dtr = xgb.QuantileDMatrix(_Iter(tr_parts), max_bin=256)
        dva = xgb.QuantileDMatrix(X[bounds[k]:bounds[k+1]], y[bounds[k]:bounds[k+1]],
                                  weight=w[bounds[k]:bounds[k+1]], ref=dtr)
        m = xgb.train(params, dtr, rounds, evals=[(dva, "valid")], early_stopping_rounds=100, verbose_eval=False)
        m.feature_names = feats
        models.append(m); del dtr, dva; gc.collect()
        log(f"   fold {k}: best_it={m.best_iteration} ({time.time()-t:.0f}s)")
    return models

def predict_split(sp_, models, feats, tag, extra=()):
    """OOF for train (each S1 by the model that did not see it), average of folds for test; saved as <tag>_XXXX"""
    for i in range(len(A_files(sp_))):
        D = load_chunk(sp_, i, extra); X = D[feats].to_numpy(np.float32)
        p = np.zeros(len(D))
        if sp_ == "train":
            fo = FOLD[D["li"].values]
            for k, m in enumerate(models):
                s = fo == k
                if not s.any(): continue
                Xk = X[s]
                p[s] = np.concatenate([m.inplace_predict(Xk[j:j+PRED_CHUNK], iteration_range=(0, m.best_iteration + 1))
                                       for j in range(0, len(Xk), PRED_CHUNK)])
        else:
            for m in models:
                p += np.concatenate([m.inplace_predict(X[j:j+PRED_CHUNK], iteration_range=(0, m.best_iteration + 1))
                                     for j in range(0, len(X), PRED_CHUNK)]) / len(models)
        pd.DataFrame({tag: p.astype(np.float32)}).to_parquet(f"{RUN_DIR}/{sp_}/{tag}_{i:04d}.parquet", index=False)
        del D, X; gc.collect()

probe = load_chunk("train", 0)
FEATS1 = [c for c in probe.columns if c not in ("li", "ri", "y")]
del probe
if not os.path.exists(f"{RUN_DIR}/model1_fold0.json"):
    X, y, w, f = build_train_matrix(FEATS1)
    log(f"stage-1 training rows: {len(y):,} (pos {int(y.sum()):,}) x {len(FEATS1)} features")
    models1 = train_folds(X, y, w, f, FEATS1)
    for k, m in enumerate(models1): m.save_model(f"{RUN_DIR}/model1_fold{k}.json")
    del X, y, w, f; gc.collect()
else:
    models1 = []
    for k in range(N_FOLDS):
        m = xgb.Booster(); m.load_model(f"{RUN_DIR}/model1_fold{k}.json"); m.set_param({"device": "cuda"}); models1.append(m)
    log("stage-1 models loaded from disk")
for sp_ in ("train", "test"):
    if not os.path.exists(f"{RUN_DIR}/{sp_}/p1_0000.parquet"): predict_split(sp_, models1, FEATS1, "p1")
yy = np.concatenate([pd.read_parquet(f, columns=["y"])["y"].values for f in A_files("train")])
p1 = np.concatenate([pd.read_parquet(f)["p1"].values for f in sorted(glob.glob(f"{RUN_DIR}/train/p1_*.parquet"))])
log(f"stage-1 OOF AUC={roc_auc_score(yy, p1):.5f} AP={average_precision_score(yy, p1):.5f}")

[14:16:30 +   278s] stage-1 models loaded from disk
[14:16:30 +   278s] stage-1 OOF AUC=0.99952 AP=0.99615


In [11]:
# ============================================================ 9. STAGE 2 (cluster features, parallel support) + model
def build_rv(L_all, R):
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 4), sublinear_tf=True, min_df=2, max_df=0.05, dtype=np.float32)
    vec.fit(np.concatenate([_to_np(L_all["all_phon"]), _to_np(R["all_phon"].sample(min(len(R), FIT_SAMPLE), random_state=SEED))]))
    return _transform(vec, _to_np(R["all_phon"]))

def _support(li, ri_local, pp, RVsub):
    sm = np.zeros(len(li), np.float32); sw = np.zeros(len(li), np.float32)
    bounds = np.flatnonzero(np.r_[True, li[1:] != li[:-1], True])
    for a, b in zip(bounds[:-1], bounds[1:]):
        if b - a < 2: continue
        V = RVsub[ri_local[a:b]]; S = (V @ V.T).toarray(); np.fill_diagonal(S, 0)
        q = pp[a:b]; Wm = S * q[None, :]
        sm[a:b] = Wm.max(1); sw[a:b] = Wm.sum(1) / (q.sum() - q + 1e-6)
    return sm, sw

def stage_2feats(sp_):
    files = A_files(sp_); cc = json.load(open(f"{RUN_DIR}/{sp_}/chunk_country.json"))
    if all(os.path.exists(f"{RUN_DIR}/{sp_}/S2_{i:04d}.parquet") for i in range(len(files))): return
    RV = build_rv(LALL[sp_], LR[sp_])
    for c in sorted(set(cc)):
        idx = [i for i, x in enumerate(cc) if x == c]
        t = time.time()
        D = [pd.concat([pd.read_parquet(files[i], columns=["li", "ri"]),
                        pd.read_parquet(f"{RUN_DIR}/{sp_}/p1_{i:04d}.parquet")], axis=1) for i in idx]
        n = [len(d) for d in D]; D = pd.concat(D, ignore_index=True)
        s, li, ri = D["p1"], D["li"], D["ri"]
        G = pd.DataFrame(index=D.index)
        gl = s.groupby(li)
        G["p1_max_l"] = gl.transform("max"); G["p1_sum_l"] = gl.transform("sum")
        G["p1_gap_l"] = s - G["p1_max_l"]; G["p1_rank_l"] = gl.rank(ascending=False, method="min")
        G["p1_n05_l"] = (s > 0.5).groupby(li).transform("sum")
        mx = s.groupby(ri).transform("max"); sec = np.maximum(second_max(s.values, ri.values), 0)
        G["p1_gap_r"] = np.where(s >= mx, s - sec, s - mx); G["p1_n05_r"] = (s > 0.5).groupby(ri).transform("sum")
        # support (parallel over S1 groups; rows are grouped by li inside each chunk)
        o = np.argsort(li.values, kind="stable"); lis = li.values[o]
        cuts = np.flatnonzero(np.r_[True, lis[1:] != lis[:-1], True])     # group boundaries (incl. end)
        jobs = []
        for g in np.array_split(np.arange(len(cuts) - 1), N_JOBS * 4):
            if len(g) == 0: continue
            a, b = cuts[g[0]], cuts[g[-1] + 1]
            rows = o[a:b]; uniq, loc = np.unique(ri.values[rows], return_inverse=True)
            jobs.append((rows, lis[a:b], loc, s.values[rows], RV[uniq]))
        res = Parallel(n_jobs=N_JOBS)(delayed(_support)(l_, loc_, p_, rv_) for _, l_, loc_, p_, rv_ in jobs)
        smax = np.zeros(len(D), np.float32); sw = np.zeros(len(D), np.float32)
        for (rows, *_), (a_, b_) in zip(jobs, res): smax[rows] = a_; sw[rows] = b_
        G["sup_max"] = smax; G["sup_w"] = sw
        G = G.astype(np.float32)
        st = 0
        for i, k in zip(idx, n):
            G.iloc[st:st + k].reset_index(drop=True).to_parquet(f"{RUN_DIR}/{sp_}/S2_{i:04d}.parquet", index=False); st += k
        log(f"   {sp_} stage-2 features [{c}] ({time.time()-t:.0f}s)")
        del D, G; gc.collect()
    del RV; gc.collect()

for sp_ in ("train", "test"): stage_2feats(sp_)
FEATS2 = FEATS1 + ["p1", "p1_max_l", "p1_sum_l", "p1_gap_l", "p1_rank_l", "p1_n05_l", "p1_gap_r", "p1_n05_r", "sup_max", "sup_w"]
if not os.path.exists(f"{RUN_DIR}/model2_fold0.json"):
    X, y, w, f = build_train_matrix(FEATS2, extra=("p1", "S2"))
    models2 = train_folds(X, y, w, f, FEATS2)
    for k, m in enumerate(models2): m.save_model(f"{RUN_DIR}/model2_fold{k}.json")
    del X, y, w, f; gc.collect()
else:
    models2 = []
    for k in range(N_FOLDS):
        m = xgb.Booster(); m.load_model(f"{RUN_DIR}/model2_fold{k}.json"); m.set_param({"device": "cuda"}); models2.append(m)
for sp_ in ("train", "test"):
    if not os.path.exists(f"{RUN_DIR}/{sp_}/p2_0000.parquet"): predict_split(sp_, models2, FEATS2, "p2", extra=("p1", "S2"))
p2 = np.concatenate([pd.read_parquet(f)["p2"].values for f in sorted(glob.glob(f"{RUN_DIR}/train/p2_*.parquet"))])
log(f"stage-2 OOF AUC={roc_auc_score(yy, p2):.5f} AP={average_precision_score(yy, p2):.5f}")

[14:16:31 +   279s] stage-2 OOF AUC=0.99949 AP=0.99588


In [12]:
# ============================================================ 10. DECISION LAYER (tuned on all train OOF, exact macro F0.5)
def load_pairs(sp_):
    li = np.concatenate([pd.read_parquet(f, columns=["li"])["li"].values for f in A_files(sp_)])
    ri = np.concatenate([pd.read_parquet(f, columns=["ri"])["ri"].values for f in A_files(sp_)])
    p = np.concatenate([pd.read_parquet(f)["p2"].values for f in sorted(glob.glob(f"{RUN_DIR}/{sp_}/p2_*.parquet"))])
    return li, ri, p.astype(np.float64)

def one_to_one(ri, p):
    best = pd.Series(p).groupby(ri).transform("max").values
    return np.where(p >= best - 1e-12, p, 0.0)

def f05_entities(li, keep, y, true_n):
    k = np.bincount(li[keep], minlength=len(true_n)).astype(np.float64)
    tp = np.bincount(li[keep], weights=y[keep], minlength=len(true_n))
    F = np.where(tp > 0, 1.25 * tp / (0.25 * true_n + k), 0.0); F[(true_n == 0) & (k == 0)] = 1.0
    return F, k, tp

def group_max(li, p): return pd.Series(p).groupby(li).transform("max").values
def mask_thresh(li, p, tg, ti, mx=None):
    mx = group_max(li, p) if mx is None else mx
    return (mx >= tg) & ((p >= ti) | (p >= mx))

def _exp_part(pp_list, n_samp=400, seed=0):
    rng = np.random.default_rng(seed); out = []
    for pp in pp_list:
        if len(pp) == 0 or pp[0] < 0.02: out.append(0); continue
        pp = pp[:30]; S = rng.random((n_samp, len(pp))) < pp[None, :]
        T = S.sum(1); e = (1.25 * np.cumsum(S, 1) / (0.25 * T[:, None] + np.arange(1, len(pp) + 1)[None, :])).mean(0)
        kb = int(np.argmax(e)); out.append(kb + 1 if e[kb] > np.mean(T == 0) else 0)
    return out

def mask_expected(li, p):
    o = np.lexsort((-p, li)); lis = li[o]; ps = p[o]
    cuts = np.flatnonzero(np.r_[True, lis[1:] != lis[:-1], True])
    groups = [ps[a:b] for a, b in zip(cuts[:-1], cuts[1:])]
    parts = np.array_split(np.arange(len(groups)), N_JOBS * 4)
    ks = Parallel(n_jobs=N_JOBS)(delayed(_exp_part)([groups[j] for j in part], seed=int(part[0]) if len(part) else 0)
                                  for part in parts)
    ks = np.concatenate([np.array(k, int) for k in ks])
    keep = np.zeros(len(p), bool)
    for (a, b), kk in zip(zip(cuts[:-1], cuts[1:]), ks):
        if kk: keep[o[a:a + kk]] = True
    return keep

def fit_iso(p, y, n=20_000_000):
    idx = np.random.default_rng(SEED).choice(len(p), min(n, len(p)), replace=False)
    return IsotonicRegression(out_of_bounds="clip").fit(p[idx], y[idx])

li_tr, ri_tr, p_tr = load_pairs("train")
results = []
for use121 in (False, True):
    p = one_to_one(ri_tr, p_tr) if use121 else p_tr
    best = (0, None); mx = group_max(li_tr, p)
    for tg in np.arange(0.20, 0.91, 0.05):
        for ti in np.arange(0.10, tg + 1e-9, 0.05):
            sc = f05_entities(li_tr, mask_thresh(li_tr, p, tg, ti, mx), yy, TRUE_N)[0].mean()
            if sc > best[0]: best = (sc, (float(round(tg, 2)), float(round(ti, 2))))
    results.append(("thresh", use121, best[1], best[0])); log(f"thresh   one2one={use121}: {best[0]:.5f} @ {best[1]}")
    iso = fit_iso(p, yy)
    sc = f05_entities(li_tr, mask_expected(li_tr, iso.predict(p)), yy, TRUE_N)[0].mean()
    results.append(("expected", use121, None, sc)); log(f"expected one2one={use121}: {sc:.5f}")
best_cfg = max(results, key=lambda x: x[3])
log("BEST decision rule:", best_cfg)

def apply_rule(li, ri, p_raw, cfg, p_ref, y_ref):
    kind, use121, prm, _ = cfg
    p = one_to_one(ri, p_raw) if use121 else p_raw
    if kind == "thresh": return mask_thresh(li, p, *prm)
    return mask_expected(li, fit_iso(p_ref, y_ref).predict(p))

p_ref = one_to_one(ri_tr, p_tr) if best_cfg[1] else p_tr
keep_tr = apply_rule(li_tr, ri_tr, p_tr, best_cfg, p_ref, yy)
F, k, tp = f05_entities(li_tr, keep_tr, yy, TRUE_N)
F_or = f05_entities(li_tr, yy > 0, yy, TRUE_N)[0]
cn = LSUB["train"]["country_n"].values
summary = {"f05": F.mean(), "precision": tp.sum() / max(k.sum(), 1), "recall": tp.sum() / TRUE_N.sum(),
           "oracle_same_shortlist": F_or.mean(), "blocking_recall": yy.sum() / TRUE_N.sum(),
           "singletons": F[TRUE_N == 0].mean(), "non_singletons": F[TRUE_N > 0].mean(),
           **{f"f05_{c}": F[cn == c].mean() for c in np.unique(cn)}}
log("VALIDATION (OOF, all train S1):", {k_: round(float(v), 4) for k_, v in summary.items()})

[14:16:31 +   279s] thresh   one2one=False: 0.97201 @ (0.8, 0.8)
[14:16:32 +   280s] expected one2one=False: 0.97270
[14:16:33 +   280s] thresh   one2one=True: 0.97201 @ (0.8, 0.8)
[14:16:33 +   281s] expected one2one=True: 0.97274
[14:16:33 +   281s] BEST decision rule: ('expected', True, None, np.float64(0.9727440546448847))
[14:16:34 +   281s] VALIDATION (OOF, all train S1): {'f05': 0.9727, 'precision': 0.9935, 'recall': 0.9432, 'oracle_same_shortlist': 0.9957, 'blocking_recall': 0.9907, 'singletons': 0.9509, 'non_singletons': 0.9741, 'f05_india': 0.0, 'f05_us': 0.9735}


In [13]:
# ============================================================ 11. TEST: apply, write, validate, record
li_te, ri_te, p_te = load_pairs("test")
keep_te = apply_rule(li_te, ri_te, p_te, best_cfg, p_ref, yy)
Lte, Rte = LSUB["test"], LR["test"]
ids_l, ids_r = _to_np(Lte["entity_id"]), _to_np(Rte["entity_id"])   # plain numpy: per-row lookups on Arrow arrays are ~1000x slower
pred, cand = defaultdict(list), defaultdict(list)
for l, r in zip(li_te, ri_te): cand[l].append(r)
for l, r in zip(li_te[keep_te], ri_te[keep_te]): pred[l].append(r)

def write_ids(path, colname, m):
    with open(path, "w", encoding="utf-8", newline="") as f:
        f.write(f"source1_entity_id\t{colname}\n")
        for l, s in enumerate(ids_l):
            f.write(f"{s}\t{','.join(sorted(set(ids_r[m.get(l, [])])))}\n")
write_ids(f"{OUT_DIR}/matching_results.tsv", "matched_entity_ids", pred)
write_ids(f"{OUT_DIR}/candidate_pairs.tsv", "candidate_entity_ids", cand)
nonempty = np.mean([len(pred.get(l, [])) > 0 for l in range(len(ids_l))])
log(f"WROTE {OUT_DIR}: S1={len(ids_l):,} non-empty={nonempty:.3f} mean matches={keep_te.sum()/len(ids_l):.2f}")
json.dump({"run": os.path.basename(RUN_DIR), "validation": {k_: float(v) for k_, v in summary.items()},
           "decision": str(best_cfg), "train_rows_rule": f"all positives + negatives rank<{TRAIN_NEG_TOP} + {TRAIN_NEG_REST:.0%} rest (weighted)",
           "test": {"s1": len(ids_l), "non_empty": float(nonempty), "mean_matches": float(keep_te.sum() / len(ids_l)),
                    "pairs": int(len(li_te))}},
          open(f"{RUN_DIR}/metrics.json", "w"), indent=1)
if not DEV:
    r = subprocess.run([sys.executable, "utils/validate_submission.py", "--matching", f"{OUT_DIR}/matching_results.tsv",
                        "--candidate", f"{OUT_DIR}/candidate_pairs.tsv", "--test-dir", "dataset/test"],
                       capture_output=True, text=True, cwd=ROOT)
    print(r.stdout[-2500:], r.stderr[-1500:])

[14:16:38 +   286s] WROTE C:/Users/malla/ml_challenghe/output_dev: S1=75,664 non-empty=0.950 mean matches=3.32
